In [ ]:
import cdfmm
import numpy as np
import time
import fmm3dpy

In [ ]:
# ==================================================================
# Physical problem setup
# ==================================================================

N = 100000                   # Number of dipoles
L_nm = 100.0               # Side length of the simulation cube [nm]

mu0_Ms = 1.5               # Saturation magnetization expressed as mu0*Ms [T]

moment_scale_min = 0.5     # 0.5 -> half the reference cell moment
moment_scale_max = 1.5     # 1.5 -> 1.5 times the reference cell moment

rng = np.random.default_rng(seed=42)


fmm3d_tol = 1e-3

In [ ]:
# ------------------------------------------------------------------
# Convert to SI units
# ------------------------------------------------------------------

mu0 = 4.0 * np.pi * 1e-7          # Vacuum permeability [T m / A]

L = L_nm * 1e-9                    # Simulation side length [m]

Ms = mu0_Ms / mu0                  # Saturation magnetization [A/m]


# ------------------------------------------------------------------
# Equivalent micromagnetic cell
# ------------------------------------------------------------------

# Imagine that the full volume L^3 were divided evenly between N
# uniformly magnetized cubic cells.
#
# Each cell would then have volume
#
#       V_cell = L^3 / N
#
cell_volume = L**3 / N             # [m^3]

# Equivalent cube side length
cell_size = cell_volume**(1.0 / 3.0)   # [m]

# A uniformly magnetized cell with magnetization Ms has dipole moment
#
#       m_ref = Ms * V_cell
#
# This defines "moment strength = 1".
#
moment_ref = Ms * cell_volume       # [A m^2]
# ==================================================================
# Random dipole positions
# ==================================================================

# Random positions distributed uniformly through the simulation cube:
#
#   [-L/2, L/2] x [-L/2, L/2] x [-L/2, L/2]
#
positions = rng.uniform(
    -L / 2,
    L / 2,
    size=(N, 3),
)
# ==================================================================
# Random dipole moments
# ==================================================================

# Generate isotropically distributed random directions.
directions = rng.normal(size=(N, 3))
directions /= np.linalg.norm(directions, axis=1)[:, None]


# Random moment strengths between 0.5 and 1.5.
#
# A strength of 1 corresponds to:
#
#       |m| = Ms * V_cell
#
# i.e. the moment of one uniformly magnetized equivalent cell at
# saturation magnetization Ms.
#
moment_scales = rng.uniform(
    moment_scale_min,
    moment_scale_max,
    size=N,
)

moments = directions * (moment_ref * moment_scales)[:, None]

In [ ]:

# ==================================================================
# FMM options
# ==================================================================

options = cdfmm.UniformFmmOptions()
options.fixed_target_source_indices = np.arange(N, dtype=int).tolist()


# ------------------------------------------------------------------
# Expansion order
# ------------------------------------------------------------------

# Maximum total degree of the Cartesian multipole expansion.
#
# Available choices:
#   Any integer >= 0
#
# Higher order:
#   + Higher accuracy
#   - More coefficients and more computational work
#
options.expansion_order = 6


# ------------------------------------------------------------------
# Tree depth
# ------------------------------------------------------------------

# Maximum level of the uniform octree.
#
# Available choices:
#   Any integer >= 0
#
#   0 = root box only
#   1 = root + 8 child boxes
#   2 = root + two subdivision levels
#   ...
#
# Deeper trees:
#   + Fewer particles in each leaf / less direct near-field work
#   - More tree boxes and FMM translations
#
options.tree.max_level = 4


# ------------------------------------------------------------------
# Empty nodes
# ------------------------------------------------------------------

# Whether empty boxes should be included.
#
# Available choices:
#   True
#   False
#
# CURRENT IMPLEMENTATION:
#   All boxes are currently materialised regardless of this setting.
#
options.tree.include_empty_nodes = True


# ------------------------------------------------------------------
# Root box shape
# ------------------------------------------------------------------

# Whether the root box should be cubic.
#
# Available choices:
#   True
#   False
#
# CURRENT IMPLEMENTATION:
#   Only cubic root boxes are currently supported.
#
options.tree.cubic_root_box = True


# ------------------------------------------------------------------
# Root box centre
# ------------------------------------------------------------------

# Available choices:
#   None                    -> automatically determine from the positions
#   cdfmm.Vec3(x, y, z)    -> explicitly specify the centre
#
# Since the random positions were generated inside a cube centred at zero,
# we specify the centre explicitly here.
#
options.tree.root_centre = None
#options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)


# ------------------------------------------------------------------
# Root box half-width
# ------------------------------------------------------------------

# Available choices:
#   None           -> automatically determine from the positions
#   positive float -> explicitly specify the half-width
#
# The box has side length L, so its half-width is L/2.
#

options.tree.root_half_width = None
#options.tree.root_half_width = L / 2



# ------------------------------------------------------------------
# M2L implementation
# ------------------------------------------------------------------

# Available choices:
#
#   cdfmm.M2LBackend.Static
#       Uses precomputed/cached M2L translation matrices.
#       This is the normal optimized implementation.
#
#   cdfmm.M2LBackend.Reference
#       Uses the reference M2L implementation.
#       Mainly intended for validation/testing.
#
options.m2l_backend = cdfmm.M2LBackend.Static


# ------------------------------------------------------------------
# Static matrix multiplication backend
# ------------------------------------------------------------------

# Available choices:
#
#   cdfmm.StaticMatrixBackend.PORTABLE
#       Portable CPU implementation.
#
#   cdfmm.StaticMatrixBackend.ONE_MKL
#       Uses Intel oneMKL for grouped M2L matrix multiplication.
#       Requires the library to have been built with oneMKL support.
#
options.static_matrix_backend = cdfmm.StaticMatrixBackend.ONE_MKL


# ------------------------------------------------------------------
# Complete FMM execution backend
# ------------------------------------------------------------------

# Available choices:
#
#   cdfmm.ExecutionBackend.AUTO
#       Automatically selects a backend.
#       Currently resolves conservatively to CPU_STATIC.
#
#   cdfmm.ExecutionBackend.CPU_REFERENCE
#       Reference CPU FMM implementation.
#       Mainly for validation/testing.
#
#   cdfmm.ExecutionBackend.CPU_STATIC
#       Optimized static CPU FMM.
#
#   cdfmm.ExecutionBackend.CUDA_PARTIAL
#       P2M/M2M/L2L/L2P on CPU.
#       M2L and P2P on GPU.
#
#   cdfmm.ExecutionBackend.CUDA_FULL
#       Complete FMM evaluation on the GPU.
#
# Compatibility aliases for CUDA_PARTIAL also exist:
#   CUDA_M2L_P2P
#   CUDA_M2L
#   CUDA_M2L_STATIC_P2P
#
options.backend = cdfmm.ExecutionBackend.CUDA_FULL

In [ ]:
# ==================================================================
# Create FMM
# ==================================================================

# We evaluate the field at the dipole positions themselves, so positions
# are both the source and target coordinates.

start = time.perf_counter()

fmm = cdfmm.UniformFmm(
    positions,
    positions,
    options,
)

fmm_setup_time = time.perf_counter() - start

In [ ]:
# ==================================================================
# FMM evaluation
# ==================================================================

# Target i is the same physical particle as source i.
# This tells the FMM to exclude self-interaction.
self_indices = np.arange(N)

start = time.perf_counter()

result = fmm.evaluate(
    moments,
    output="field",
    target_source_indices=self_indices,
)

fmm_evaluation_time = time.perf_counter() - start

H_fmm = result["H"]

In [ ]:
# ==================================================================
# FMM3D input preparation
# ==================================================================

# FMM3D expects arrays with shape:
#
#       positions: (3, N)
#       dipoles:   (3, N)
#
# whereas cdfmm uses (N, 3).
#
# We evaluate at the source locations themselves, just as for cdfmm.
# FMM3D automatically omits the self-interaction at source locations.
#
fmm3d_sources = np.ascontiguousarray(positions.T)
fmm3d_dipvec = np.ascontiguousarray(moments.T)


In [ ]:
# ==================================================================
# FMM3D evaluation
# ==================================================================

start = time.perf_counter()

result_fmm3d = fmm3dpy.lfmm3d(
    eps=fmm3d_tol,
    sources=fmm3d_sources,
    dipvec=fmm3d_dipvec,

    # pg = 2 requests both potential and gradient at the source locations.
    pg=2,

    # No separate target locations are used here.
    pgt=0,
)

fmm3d_evaluation_time = time.perf_counter() - start


# FMM3D returns grad(phi), while cdfmm defines the magnetic field as
#
#       H = -grad(phi)
#
# Transpose back from FMM3D's (3, N) layout to cdfmm's (N, 3) layout.
#
H_fmm3d = -result_fmm3d.grad.T


In [ ]:
# FMM3D evaluation is performed in the previous cell.


In [ ]:
# ==================================================================
# Direct evaluation
# ==================================================================

# The direct method evaluates every target against every source:
#
#       O(N^2)
#
# We test both available direct implementations:
#
#   CPU:
#       cdfmm.direct_p2p_reference(...)
#
#   GPU:
#       cdfmm.cuda_direct_p2p_reference(...)
#
# Both calculate the same direct dipole-dipole interaction.
# The GPU version requires CUDA support and an available CUDA device.


# ------------------------------------------------------------------
# CPU direct evaluation
# ------------------------------------------------------------------

start = time.perf_counter()

result_direct_cpu = cdfmm.direct_p2p_reference(
    positions,
    positions,
    moments,
    output="field",
    target_source_indices=self_indices,
)

direct_cpu_time = time.perf_counter() - start

H_direct_cpu = result_direct_cpu["H"]


# ------------------------------------------------------------------
# GPU direct evaluation
# ------------------------------------------------------------------

H_direct_gpu = None
direct_gpu_time = None

if cdfmm.cuda_direct_available():

    start = time.perf_counter()

    result_direct_gpu = cdfmm.cuda_direct_p2p_reference(
        positions,
        positions,
        moments,
        output="field",
        target_source_indices=self_indices,
    )

    direct_gpu_time = time.perf_counter() - start

    H_direct_gpu = result_direct_gpu["H"]

else:
    print("CUDA direct evaluation is not available.")

In [ ]:
# ==================================================================
# Accuracy comparison
# ==================================================================

# Use the CPU direct calculation as the reference solution.
#
# We compare:
#   1. cdfmm FMM  vs CPU direct
#   2. FMM3D      vs CPU direct
#   3. GPU direct vs CPU direct   (when CUDA direct is available)
#
# For each comparison we calculate:
#   - pointwise absolute field error
#   - pointwise relative field error
#   - global relative L2 error
#   - RMS error over all field components


def field_error_metrics(H, H_reference):
    error = H - H_reference

    # Absolute vector error at each target
    absolute_error = np.linalg.norm(error, axis=1)

    # Magnitude of the reference field at each target
    reference_magnitude = np.linalg.norm(H_reference, axis=1)

    # Pointwise relative vector error.
    # The epsilon prevents division by zero for vanishing reference fields.
    relative_error = absolute_error / np.maximum(
        reference_magnitude,
        np.finfo(float).eps,
    )

    # Global relative L2 error
    relative_l2_error = (
        np.linalg.norm(error)
        / np.linalg.norm(H_reference)
    )

    # RMS error over all field components
    rmse = np.sqrt(np.mean(error**2))

    return {
        "mean_absolute_error": np.mean(absolute_error),
        "max_absolute_error": np.max(absolute_error),
        "mean_relative_error": np.mean(relative_error),
        "max_relative_error": np.max(relative_error),
        "relative_l2_error": relative_l2_error,
        "rmse": rmse,
    }


# FMM accuracy relative to the CPU direct reference
fmm_error = field_error_metrics(
    H_fmm,
    H_direct_cpu,
)


# FMM3D accuracy relative to the CPU direct reference
fmm3d_error = field_error_metrics(
    H_fmm3d,
    H_direct_cpu,
)


# GPU direct accuracy relative to the CPU direct reference
gpu_direct_error = None

if H_direct_gpu is not None:
    gpu_direct_error = field_error_metrics(
        H_direct_gpu,
        H_direct_cpu,
    )


In [ ]:
# ==================================================================
# Results
# ==================================================================

fmm_total_time = fmm_setup_time + fmm_evaluation_time


# ------------------------------------------------------------------
# Problem information
# ------------------------------------------------------------------

print()
print("Physical problem")
print("----------------")
print(f"N:                        {N}")
print(f"Simulation side length:   {L_nm:.1f} nm")
print(f"mu0 Ms:                   {mu0_Ms:.3f} T")
print(f"Ms:                       {Ms:.6e} A/m")
print(f"Equivalent cell size:     {cell_size * 1e9:.3f} nm")
print(f"Reference dipole moment:  {moment_ref:.6e} A m^2")
print(
    f"Moment scale range:        "
    f"{moment_scale_min:.2f} - {moment_scale_max:.2f}"
)

print()
print("FMM parameters")
print("--------------")
print(f"Expansion order:           {options.expansion_order}")
print(f"Tree depth:                {options.tree.max_level}")
print(f"FMM3D tolerance:           {fmm3d_tol:.1e}")


# ------------------------------------------------------------------
# Timing
# ------------------------------------------------------------------

print()
print("Timing")
print("------")
print(f"FMM setup:                 {fmm_setup_time:.6f} s")
print(f"FMM evaluation:            {fmm_evaluation_time:.6f} s")
print(f"FMM setup + evaluation:    {fmm_total_time:.6f} s")
print(f"FMM3D evaluation:          {fmm3d_evaluation_time:.6f} s")
print(f"CPU direct:                {direct_cpu_time:.6f} s")

if direct_gpu_time is not None:
    print(f"GPU direct:                {direct_gpu_time:.6f} s")


# ------------------------------------------------------------------
# Speedup
# ------------------------------------------------------------------

print()
print("Speedup")
print("-------")

print(
    f"FMM eval vs CPU direct:    "
    f"{direct_cpu_time / fmm_evaluation_time:.2f}x"
)

print(
    f"FMM total vs CPU direct:   "
    f"{direct_cpu_time / fmm_total_time:.2f}x"
)


print(
    f"FMM3D vs CPU direct:       "
    f"{direct_cpu_time / fmm3d_evaluation_time:.2f}x"
)

print(
    f"cdfmm eval vs FMM3D:       "
    f"{fmm3d_evaluation_time / fmm_evaluation_time:.2f}x"
)

print(
    f"cdfmm total vs FMM3D:      "
    f"{fmm3d_evaluation_time / fmm_total_time:.2f}x"
)

if direct_gpu_time is not None:
    print(
        f"GPU direct vs CPU direct:  "
        f"{direct_cpu_time / direct_gpu_time:.2f}x"
    )

    print(
        f"FMM eval vs GPU direct:    "
        f"{direct_gpu_time / fmm_evaluation_time:.2f}x"
    )

    print(
        f"FMM total vs GPU direct:   "
        f"{direct_gpu_time / fmm_total_time:.2f}x"
    )


# ------------------------------------------------------------------
# Accuracy: FMM vs CPU direct
# ------------------------------------------------------------------

print()
print("Accuracy: FMM vs CPU direct")
print("---------------------------")
print(
    f"Relative L2 error:         "
    f"{fmm_error['relative_l2_error']:.6e}"
)
print(
    f"Mean relative error:       "
    f"{fmm_error['mean_relative_error']:.6e}"
)
print(
    f"Maximum relative error:    "
    f"{fmm_error['max_relative_error']:.6e}"
)
print(
    f"RMSE:                      "
    f"{fmm_error['rmse']:.6e} A/m"
)
print(
    f"Mean absolute error:       "
    f"{fmm_error['mean_absolute_error']:.6e} A/m"
)
print(
    f"Maximum absolute error:    "
    f"{fmm_error['max_absolute_error']:.6e} A/m"
)


# ------------------------------------------------------------------
# Accuracy: FMM3D vs CPU direct
# ------------------------------------------------------------------

print()
print("Accuracy: FMM3D vs CPU direct")
print("-----------------------------")
print(
    f"Relative L2 error:         "
    f"{fmm3d_error['relative_l2_error']:.6e}"
)
print(
    f"Mean relative error:       "
    f"{fmm3d_error['mean_relative_error']:.6e}"
)
print(
    f"Maximum relative error:    "
    f"{fmm3d_error['max_relative_error']:.6e}"
)
print(
    f"RMSE:                      "
    f"{fmm3d_error['rmse']:.6e} A/m"
)
print(
    f"Mean absolute error:       "
    f"{fmm3d_error['mean_absolute_error']:.6e} A/m"
)
print(
    f"Maximum absolute error:    "
    f"{fmm3d_error['max_absolute_error']:.6e} A/m"
)



# ------------------------------------------------------------------
# Accuracy: GPU direct vs CPU direct
# ------------------------------------------------------------------

if gpu_direct_error is not None:

    print()
    print("Accuracy: GPU direct vs CPU direct")
    print("----------------------------------")
    print(
        f"Relative L2 difference:    "
        f"{gpu_direct_error['relative_l2_error']:.6e}"
    )
    print(
        f"Mean relative difference:  "
        f"{gpu_direct_error['mean_relative_error']:.6e}"
    )
    print(
        f"Maximum relative diff.:    "
        f"{gpu_direct_error['max_relative_error']:.6e}"
    )
    print(
        f"RMSE:                      "
        f"{gpu_direct_error['rmse']:.6e} A/m"
    )
